In [1]:
import os
import pandas as pd
import numpy as np
import ast

import sys
sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config

In [2]:
lake = 'zug'
folder_path = rf"/storage/alplakes_test/{lake}_100m_2025"
input_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_final")

In [3]:
date_range = [pd.to_datetime('2025-03-01'), pd.to_datetime('2026-03-01')]

# Import catalogue

In [4]:
df_lake = pd.read_csv(os.path.join(input_folder, "lake_characteristics.csv"))
df_lake = df_lake.set_index('id', drop=False)
df_lake['date'] = pd.to_datetime(df_lake['date'])
df_lake = df_lake[df_lake['date'] >= date_range[0]]

In [5]:
df_lvl0 = pd.read_csv(os.path.join(input_folder, 'lvl0.csv'))
df_lvl0 = df_lvl0.set_index('id', drop=False)
df_lvl0['date'] = pd.to_datetime(df_lvl0['date'])
df_lvl0 = df_lvl0[df_lvl0['date'].between(*date_range, inclusive='left')].copy()


In [6]:
lake_mask = np.load(os.path.join(folder_path, "grid", "mask_lake.npy"))
depths = pd.read_csv(os.path.join(folder_path, "grid", "depths.csv"))

In [7]:
def to_array(x):
    # Catalogue CSVs use comma-separated floats, sometimes enclosed in brackets.
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') or x.startswith('('):
            try:
                x = ast.literal_eval(x)
            except (ValueError, SyntaxError):
                # Also accept NumPy-style strings such as '[1 2 3]'.
                x = x.strip('[]()')
        if isinstance(x, str):
            x = np.fromstring(x.replace(',', ' '), sep=' ')
    values = np.atleast_1d(np.asarray(x).squeeze())
    if values.ndim != 1:
        raise ValueError(f'Expected a vector of cell indices, got shape {values.shape}')
    if np.issubdtype(values.dtype, np.integer):
        return values.astype(np.intp, copy=False)
    values = values.astype(float, copy=False)
    if not np.all(np.isfinite(values)) or np.any(values != np.floor(values)):
        raise ValueError('Eddy cell indices must be finite integers')
    return values.astype(np.intp)


In [8]:
df_lvl0['i_eddy_cells'] = df_lvl0['i_eddy_cells'].apply(lambda s: to_array(s).astype(int))
df_lvl0['j_eddy_cells'] = df_lvl0['j_eddy_cells'].apply(lambda s: to_array(s).astype(int))

# Import MITgcm results

In [9]:
model = f'{lake}_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

# Get velocities per eddy

Load all required depths once per timestamp and use paired NumPy indexing (`u[j, i]`).
Reduce all eddy cells together per depth. Set `n_workers` in the calculation cell
to compare sequential processing with 2 or 4 parallel timestamp reads; each worker
holds one timestamp of the required U/V depths plus one depth of eddy cells.
U and V are averaged from adjacent MITgcm faces onto cell centres.
Catalogue `xc`, `yc` are fractional grid indices; distances use the regular
Cartesian grid spacing from `dxF`, `dyF` (metres).

The maximum is the largest **absolute tangential velocity** among eddy cells,
not an annular average. Its signed value is positive anticlockwise; the radius
is the distance from the catalogue centre to that same cell. Centre cells,
dry cells, missing velocities and unavailable outer faces are excluded.
Dates are matched exactly to model timestamps, avoiding assumptions about
whether catalogue `time_index` starts at zero or one.


In [23]:
def regular_spacing(ds, name):
    values = np.asarray(ds[name].values, dtype=float)
    valid = values[np.isfinite(values) & (values > 0)]
    if not valid.size or not np.allclose(valid, valid[0], rtol=1e-5):
        raise ValueError(f'{name}: this calculation requires a regular Cartesian grid')
    return float(valid[0])


def eddy_velocity_profile(u, v, i, j, xc, yc, dx, dy):
    """Paired cell velocities and tangential projection about the eddy centre."""
    # Normalize scalar, list and CSV indices, including single-cell eddies.
    i, j = to_array(i), to_array(j)
    if i.shape != j.shape or i.ndim != 1:
        raise ValueError(f'Unpaired eddy indices: i shape={i.shape}, j shape={j.shape}; '
                         f'i sample={i[:5]}, j sample={j[:5]}')
    if np.any((i < 0) | (i >= u.shape[1]) | (j < 0) | (j >= u.shape[0])):
        raise IndexError('Eddy cell outside the model grid')
    u_cells, v_cells = u[j, i], v[j, i]
    x, y = (i - xc) * dx, (j - yc) * dy
    radius = np.hypot(x, y)
    azimuthal = np.full(radius.shape, np.nan)
    valid = (radius > 0) & np.isfinite(u_cells) & np.isfinite(v_cells)
    np.divide(x * v_cells - y * u_cells, radius, out=azimuthal, where=valid)
    return u_cells, v_cells, radius, azimuthal


def centred_velocity_slice(ds, time_index, depth_index):
    """Read one U/V slice, then centre it without xarray coordinate alignment."""
    raw_u = ds.UVEL.isel(time=time_index, Z=depth_index).transpose('YC', 'XG').values
    raw_v = ds.VVEL.isel(time=time_index, Z=depth_index).transpose('YG', 'XC').values
    ny, nx = ds.sizes['YC'], ds.sizes['XC']
    u, v = np.full((ny, nx), np.nan), np.full((ny, nx), np.nan)
    n_u, n_v = min(nx, raw_u.shape[1] - 1), min(ny, raw_v.shape[0] - 1)
    u[:, :n_u] = 0.5 * (raw_u[:, :n_u] + raw_u[:, 1:n_u + 1])
    v[:n_v, :] = 0.5 * (raw_v[:n_v, :] + raw_v[1:n_v + 1, :])
    wet = ds.hFacC.isel(Z=depth_index).transpose('YC', 'XC').values > 0
    u[~wet], v[~wet] = np.nan, np.nan
    return u, v


def maxima_for_depth(u, v, positions, indices_i, indices_j, centres_x, centres_y, ids, dx, dy):
    """Vectorize all eddy cells in a depth slice; preserve the first maximum on ties."""
    ii, jj = [], []
    for position in positions:
        try:
            i, j = to_array(indices_i[position]), to_array(indices_j[position])
            if i.shape != j.shape:
                raise ValueError(f'Unpaired index shapes: {i.shape}, {j.shape}')
            if np.any((i < 0) | (i >= u.shape[1]) | (j < 0) | (j >= u.shape[0])):
                raise IndexError('Cell outside the model grid')
        except (ValueError, TypeError, IndexError) as exc:
            raise ValueError(f'Eddy id={ids[position]}: {exc}') from exc
        ii.append(i)
        jj.append(j)
    lengths = np.array([len(i) for i in ii], dtype=np.intp)
    output = np.full((len(positions), 5), np.nan)
    nonempty = lengths > 0
    if not nonempty.any():
        return output
    i, j = np.concatenate(ii), np.concatenate(jj)
    x = (i - np.repeat(centres_x[positions], lengths)) * dx
    y = (j - np.repeat(centres_y[positions], lengths)) * dy
    radius = np.hypot(x, y)
    uc, vc = u[j, i], v[j, i]
    azimuthal = np.full(radius.shape, np.nan)
    valid = (radius > 0) & np.isfinite(uc) & np.isfinite(vc)
    np.divide(x * vc - y * uc, radius, out=azimuthal, where=valid)
    speed = np.where(np.isfinite(azimuthal), np.abs(azimuthal), -np.inf)
    starts = np.cumsum(lengths) - lengths
    maxima = np.maximum.reduceat(speed, starts[nonempty])
    cell_maxima = np.repeat(maxima, lengths[nonempty])
    candidates = np.where(speed == cell_maxima, np.arange(len(i)), len(i))
    winners = np.minimum.reduceat(candidates, starts[nonempty])
    good = np.isfinite(maxima)
    rows, k = np.flatnonzero(nonempty)[good], winners[good]
    output[rows] = np.column_stack((speed[k], azimuthal[k], radius[k], i[k], j[k]))
    return output

from tqdm.auto import tqdm
def compute_eddy_maxima(frame, ds, dx, dy, n_workers=1):
    """Read U/V once per timestamp and reduce all cells together per depth."""
    from concurrent.futures import ThreadPoolExecutor
    from time import perf_counter
    if not isinstance(n_workers, int) or n_workers < 1:
        raise ValueError('n_workers must be a positive integer')
    model_times = pd.DatetimeIndex(ds.time.values)
    if not model_times.is_unique:
        raise ValueError('Model timestamps must be unique')
    times = model_times.get_indexer(frame['date'])
    if np.any(times < 0):
        raise ValueError('Some catalogue dates are absent from the model dataset')
    depths = frame['depth_index'].to_numpy(dtype=float)
    if np.any(~np.isfinite(depths) | (depths != np.floor(depths)) |
              (depths < 0) | (depths >= ds.sizes['Z'])):
        raise ValueError('Invalid depth indices')
    depths = depths.astype(np.intp)
    indices_i = frame['i_eddy_cells'].to_numpy()
    indices_j = frame['j_eddy_cells'].to_numpy()
    centres_x = frame['xc'].to_numpy(dtype=float)
    centres_y = frame['yc'].to_numpy(dtype=float)
    ids = frame['id'].to_numpy()
    grouped = pd.DataFrame({'t': times}).groupby('t', sort=False).indices
    ny, nx = ds.sizes['YC'], ds.sizes['XC']
    wet = ds.hFacC.transpose('Z', 'YC', 'XC').values > 0
    results = np.full((len(frame), 5), np.nan)

    def process_time(item):
        t, positions = item
        unique_depths = np.unique(depths[positions])
        # One graph evaluation per timestamp; prevent nested Dask thread pools.
        snapshot = ds[['UVEL', 'VVEL']].isel(time=int(t), Z=unique_depths)
        snapshot = snapshot.compute(scheduler='synchronous')
        raw_u = snapshot.UVEL.transpose('Z', 'YC', 'XG').values
        raw_v = snapshot.VVEL.transpose('Z', 'YG', 'XC').values
        block = np.full((len(positions), 5), np.nan)
        for depth_slot, z in enumerate(unique_depths):
            local = np.flatnonzero(depths[positions] == z)
            u, v = np.full((ny, nx), np.nan), np.full((ny, nx), np.nan)
            nu, nv = min(nx, raw_u.shape[2] - 1), min(ny, raw_v.shape[1] - 1)
            u[:, :nu] = (raw_u[depth_slot, :, :nu] + raw_u[depth_slot, :, 1:nu + 1]) * 0.5
            v[:nv, :] = (raw_v[depth_slot, :nv, :] + raw_v[depth_slot, 1:nv + 1, :]) * 0.5
            u[~wet[z]], v[~wet[z]] = np.nan, np.nan
            block[local] = maxima_for_depth(
                u, v, positions[local], indices_i, indices_j,
                centres_x, centres_y, ids, dx, dy)
        return positions, block

    started = perf_counter()

    with tqdm(total=len(grouped), desc="Processing timestamps", unit="time") as pbar:

        if n_workers == 1:
            for positions, block in map(process_time, grouped.items()):
                results[positions] = block
                pbar.update(1)

        else:
            from concurrent.futures import wait, FIRST_COMPLETED

            with ThreadPoolExecutor(max_workers=n_workers) as executor:
                items = iter(grouped.items())
                pending = set()

                for _ in range(n_workers):
                    item = next(items, None)
                    if item is not None:
                        pending.add(executor.submit(process_time, item))

                while pending:
                    done, pending = wait(pending, return_when=FIRST_COMPLETED)

                    for future in done:
                        positions, block = future.result()
                        results[positions] = block
                        pbar.update(1)

                        item = next(items, None)
                        if item is not None:
                            pending.add(executor.submit(process_time, item))

    print(f'{len(frame):,} eddies / {len(grouped):,} timestamps: '
        f'{perf_counter() - started:.2f} s ({n_workers} worker(s))')

    return results


In [11]:
df_lvl0_test = df_lvl0.iloc[[0,1,2]]

In [12]:
df_lvl0_filtered = df_lvl0[df_lvl0['kinetic_energy_eddy_[MJ]'] > 0].copy()


In [14]:
df_lvl0_filtered.head()

,id,time_index,depth_index,eddy_index,date,depth_[m],xc,yc,surface_area_[m2],volume_slice_[m3],rotation_direction,kinetic_energy_eddy_[MJ],mean_temperature_eddy_[°C],i_eddy_cells,j_eddy_cells,task_load_time_sec,task_compute_time_sec,task_total_time_sec,pixels_in_task
id,,,,,,,,,,,,,,,,,,,
1,1,1416.0,0.0,1.0,2025-03-01 00:30:00,-0.2500,39.761538,96.900000,270000.0,1.350000e+05,anticlockwise,0.018943,2.586667,"[37, 37, 37, 37, 38, 38, 38, 38, 38, 38, 38, 3...","[95, 96, 97, 98, 94, 95, 96, 97, 98, 99, 100, ...",1.345407,4.884236,6.229643,1
2,2,1416.0,1.0,0.0,2025-03-01 00:30:00,-0.7575,29.804167,23.858333,2120000.0,1.091800e+06,clockwise,0.056558,5.828977,"[23, 23, 23, 23, 23, 24, 24, 24, 24, 24, 24, 2...","[27, 28, 29, 30, 31, 24, 25, 26, 27, 28, 29, 3...",1.317651,0.330035,1.647687,1
3,3,1416.0,1.0,1.0,2025-03-01 00:30:00,-0.7575,39.351064,96.521277,190000.0,9.785000e+04,anticlockwise,0.005960,2.148757,"[37, 37, 37, 37, 38, 38, 38, 38, 38, 39, 39, 3...","[95, 96, 97, 98, 94, 95, 96, 97, 98, 94, 95, 9...",1.317651,0.330035,1.647687,1
7,7,1416.0,2.0,2.0,2025-03-01 00:30:00,-1.2805,29.764205,25.973485,2420000.0,1.285020e+06,clockwise,0.064803,5.833776,"[23, 23, 23, 23, 23, 23, 23, 24, 24, 24, 24, 2...","[27, 28, 29, 30, 31, 32, 33, 25, 26, 27, 28, 2...",1.536760,0.322553,1.859314,1
8,8,1416.0,3.0,0.0,2025-03-01 00:30:00,-1.8200,25.550000,66.193750,480000.0,2.630400e+05,anticlockwise,0.002174,5.177625,"[21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 23, 2...","[64, 65, 66, 67, 68, 65, 66, 67, 68, 69, 65, 6...",1.942598,0.385594,2.328192,1


In [24]:
dx = regular_spacing(ds, 'dxF')
dy = regular_spacing(ds, 'dyF')
model_times = pd.DatetimeIndex(ds.time.values)
columns = ['max_azimuthal_speed_[m/s]', 'azimuthal_velocity_at_max_[m/s]',
           'radius_at_max_azimuthal_speed_[m]', 'i_at_max_azimuthal_speed',
           'j_at_max_azimuthal_speed']

# Start with 2 threads. Compare 1, 2 and 4 on a small date range;
# disk throughput determines whether more workers help.
n_workers = 30
results = compute_eddy_maxima(df_lvl0_filtered, ds, dx, dy, n_workers=n_workers)
#df_lvl0_filtered[columns] = results
#df_lvl0_filtered[['id', 'date', 'depth_index'] + columns].head()


Processing timestamps:   0%|          | 0/8760 [00:00<?, ?time/s]

KeyboardInterrupt: 

In [28]:

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from time import perf_counter


def compute_eddy_maxima_fast(frame, ds, dx, dy, n_workers=4):
    """
    Compute maximum absolute azimuthal velocity for each eddy.

    Optimizations:
    - Parse eddy indices only once.
    - Precompute cell coordinates and radii.
    - Read U/V once per timestamp and required depth.
    - Interpolate velocities only at eddy cells.
    - Vectorized maximum reduction across eddies.
    - Bounded parallel execution with progress bar.

    Returns
    -------
    ndarray (n_eddies, 5)
        [max_abs_vtheta, signed_vtheta, radius, i, j]
    """

    started = perf_counter()
    n_eddies = len(frame)

    results = np.full((n_eddies, 5), np.nan)

    if n_eddies == 0:
        return results

    # --------------------------------------------------
    # 1. Match catalogue timestamps and depths
    # --------------------------------------------------

    model_times = pd.DatetimeIndex(ds.time.values)

    if not model_times.is_unique:
        raise ValueError("Model timestamps must be unique")

    time_indices = model_times.get_indexer(
        pd.DatetimeIndex(frame["date"])
    )

    if np.any(time_indices < 0):
        raise ValueError("Some catalogue dates are absent from ds")

    depth_values = frame["depth_index"].to_numpy(dtype=float)

    if np.any(
        ~np.isfinite(depth_values)
        | (depth_values != np.floor(depth_values))
        | (depth_values < 0)
        | (depth_values >= ds.sizes["Z"])
    ):
        raise ValueError("Invalid depth indices")

    depth_indices = depth_values.astype(np.intp)

    nx = ds.sizes["XC"]
    ny = ds.sizes["YC"]

    # --------------------------------------------------
    # 2. Parse all eddy cells once
    # --------------------------------------------------

    i_arrays = [
        np.asarray(to_array(x), dtype=np.intp)
        for x in frame["i_eddy_cells"]
    ]

    j_arrays = [
        np.asarray(to_array(x), dtype=np.intp)
        for x in frame["j_eddy_cells"]
    ]

    lengths = np.array(
        [len(x) for x in i_arrays], dtype=np.intp
    )

    for k, (ii, jj) in enumerate(zip(i_arrays, j_arrays)):
        if ii.ndim != 1 or jj.ndim != 1 or ii.shape != jj.shape:
            raise ValueError(
                f"Eddy id={frame['id'].iloc[k]}: invalid cell indices"
            )

        if np.any((ii < 0) | (ii >= nx) | (jj < 0) | (jj >= ny)):
            raise IndexError(
                f"Eddy id={frame['id'].iloc[k]}: cell outside grid"
            )

    total_cells = lengths.sum()

    if total_cells == 0:
        return results

    # Flatten all cells
    all_i = np.concatenate(i_arrays)
    all_j = np.concatenate(j_arrays)

    # Each cell's parent eddy
    eddy_id = np.repeat(
        np.arange(n_eddies), lengths
    )

    # Original cell offsets
    offsets = np.concatenate((
        [0], np.cumsum(lengths)
    ))

    # --------------------------------------------------
    # 3. Precompute geometry
    # --------------------------------------------------

    xc = frame["xc"].to_numpy(dtype=float)
    yc = frame["yc"].to_numpy(dtype=float)

    x = (all_i - xc[eddy_id]) * dx
    y = (all_j - yc[eddy_id]) * dy

    radius = np.hypot(x, y)

    # Tangential velocity:
    # vtheta = (-y/r)*u + (x/r)*v

    coeff_u = np.zeros(total_cells, dtype=float)
    coeff_v = np.zeros(total_cells, dtype=float)

    valid_radius = radius > 0

    coeff_u[valid_radius] = (
        -y[valid_radius] / radius[valid_radius]
    )

    coeff_v[valid_radius] = (
        x[valid_radius] / radius[valid_radius]
    )

    # --------------------------------------------------
    # 4. Group eddies by (time, depth)
    # --------------------------------------------------

    groups = {}

    for k, (t, z) in enumerate(
        zip(time_indices, depth_indices)
    ):
        groups.setdefault((int(t), int(z)), []).append(k)

    # Group depths by timestamp
    time_groups = {}

    for (t, z), positions in groups.items():
        time_groups.setdefault(t, []).append(
            (z, np.asarray(positions, dtype=np.intp))
        )

    # Load wet mask once
    wet = (
        ds.hFacC.transpose("Z", "YC", "XC").values > 0
    )

    # --------------------------------------------------
    # 5. Process one timestamp
    # --------------------------------------------------

    def process_time(item):

        t, depth_groups = item

        output_positions = []
        output_values = []

        for z, positions in depth_groups:

            # Read only one depth slice
            snapshot = ds[["UVEL", "VVEL"]].isel(
                time=t, Z=z
            ).compute(scheduler="synchronous")

            u_raw = snapshot.UVEL.transpose(
                "YC", "XG"
            ).values

            v_raw = snapshot.VVEL.transpose(
                "YG", "XC"
            ).values

            # Gather cells belonging to these eddies
            cell_indices = np.concatenate([
                np.arange(offsets[k], offsets[k + 1])
                for k in positions
            ])

            if cell_indices.size == 0:
                output_positions.extend(positions)
                output_values.extend(
                    [np.full(5, np.nan) for _ in positions]
                )
                continue

            ii = all_i[cell_indices]
            jj = all_j[cell_indices]

            # ------------------------------------------
            # Interpolate only required cells
            # ------------------------------------------

            uc = np.full(len(ii), np.nan)
            vc = np.full(len(ii), np.nan)

            valid_u = ii + 1 < u_raw.shape[1]
            valid_v = jj + 1 < v_raw.shape[0]

            uc[valid_u] = 0.5 * (
                u_raw[jj[valid_u], ii[valid_u]]
                + u_raw[jj[valid_u], ii[valid_u] + 1]
            )

            vc[valid_v] = 0.5 * (
                v_raw[jj[valid_v], ii[valid_v]]
                + v_raw[jj[valid_v] + 1, ii[valid_v]]
            )

            # Wet mask
            valid_wet = wet[z, jj, ii]

            uc[~valid_wet] = np.nan
            vc[~valid_wet] = np.nan

            # ------------------------------------------
            # Azimuthal velocity
            # ------------------------------------------

            azimuthal = (
                coeff_u[cell_indices] * uc
                + coeff_v[cell_indices] * vc
            )

            azimuthal[
                ~valid_radius[cell_indices]
            ] = np.nan

            speed = np.where(
                np.isfinite(azimuthal),
                np.abs(azimuthal),
                -np.inf
            )

            # ------------------------------------------
            # Maximum per eddy
            # ------------------------------------------

            local_lengths = lengths[positions]

            nonempty = local_lengths > 0

            starts = (
                np.cumsum(local_lengths)
                - local_lengths
            )

            maxima = np.maximum.reduceat(
                speed, starts[nonempty]
            )

            # First maximum in original cell order
            repeated_maxima = np.repeat(
                maxima, local_lengths[nonempty]
            )

            candidates = np.where(
                speed == repeated_maxima,
                np.arange(len(speed)),
                len(speed)
            )

            winners = np.minimum.reduceat(
                candidates, starts[nonempty]
            )

            block = np.full(
                (len(positions), 5), np.nan
            )

            good = np.isfinite(maxima)

            rows = np.flatnonzero(nonempty)[good]
            k = winners[good]

            block[rows, 0] = speed[k]
            block[rows, 1] = azimuthal[k]
            block[rows, 2] = radius[cell_indices[k]]
            block[rows, 3] = ii[k]
            block[rows, 4] = jj[k]

            output_positions.extend(positions)
            output_values.extend(block)

        return (
            np.asarray(output_positions, dtype=np.intp),
            np.asarray(output_values, dtype=float)
        )
    print(f'Process parallel')
    # --------------------------------------------------
    # 6. Parallel execution + progress
    # --------------------------------------------------

    items = iter(time_groups.items())

    with tqdm(
        total=len(time_groups),
        desc="Processing timestamps",
        unit="time"
    ) as pbar:

        if n_workers == 1:

            for item in items:
                positions, block = process_time(item)
                results[positions] = block
                pbar.update(1)

        else:

            with ThreadPoolExecutor(
                max_workers=n_workers
            ) as executor:

                pending = set()

                for _ in range(n_workers):
                    item = next(items, None)

                    if item is not None:
                        pending.add(
                            executor.submit(process_time, item)
                        )

                while pending:

                    done, pending = wait(
                        pending,
                        return_when=FIRST_COMPLETED
                    )

                    for future in done:

                        positions, block = future.result()

                        results[positions] = block
                        pbar.update(1)

                        item = next(items, None)

                        if item is not None:
                            pending.add(
                                executor.submit(process_time, item)
                            )

    elapsed = perf_counter() - started

    print(
        f"{n_eddies:,} eddies | "
        f"{len(time_groups):,} timestamps | "
        f"{elapsed:.1f} s | "
        f"{n_workers} workers"
    )

    return results


In [ ]:
n_workers = 4

results = compute_eddy_maxima_fast(
    df_lvl0_filtered,
    ds,
    dx,
    dy,
    n_workers=n_workers
)

df_lvl0_filtered[columns] = results

Process parallel


Processing timestamps:   0%|          | 0/8760 [00:00<?, ?time/s]